In [ ]:
# !pip install ipython==8.1.0

In [ ]:
# # Comment the following if you are running your code locally
# # This mounts your Google Drive to the Colab VM.
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder = '/content/drive/MyDrive'
# notebook_folder = drive_folder + '/project'
# %cd {notebook_folder}

# %cd ./cs682/data
# !bash prepare.sh
# %cd ../../

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import json

from models.teacher import BERTTeacher
from data.loader import IMDBDataset, YelpDataset, AmazonDataset

: 

In [ ]:
# Configuration variables
dataset = "yelp"
epochs = 3
learning_rate = 2e-5
validation_pct = 0.2
batch_size = 32
log_every_k = 50
accuracy_every_k = 200
max_length = 128
dropout = 0.1
model_save_path = "teacher.pt"
train_save_path = "teacher.json"
mapped_layer_indices = [4, 8]

num_classes = 5
if dataset == "imdb":
    data = IMDBDataset()
    num_classes = 2
elif dataset == "yelp":
    data = YelpDataset()
elif dataset == "amazon":
    data = AmazonDataset()
else:
    raise ValueError()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, tokenizer = BERTTeacher.from_pretrained(
    num_classes=num_classes,
    mapped_layer_indices=mapped_layer_indices,
    dropout=dropout,
)

val_size = int(len(data) * validation_pct)
train_size = len(data) - val_size
train_dataset, val_dataset = random_split(data, [train_size, val_size], generator=torch.Generator().manual_seed(42))

collate_fn = make_collate_fn(tokenizer, max_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

train_config = TeacherTrainConfig(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=learning_rate),
    epochs=epochs,
    log_every_k=log_every_k,
    accuracy_every_k=accuracy_every_k,
    device=device,
)

model, losses, train_accs, val_accs = train(train_config, train_loader, val_loader)

with open(train_save_path, "w") as f:
    obj = {
        "losses": losses,
        "train_accs": train_accs,
        "validation_accs": val_accs
    }
    json.dump(obj, f, indent=2)

torch.save(model.state_dict(), model_save_path)
print(f"\nModel saved to {model_save_path}")